# Lecture 1 — Missing data as Gaussian inference

This is the **solved Q&A notebook**.

We start with one known joint Gaussian, hide one block of a simulated observation, and infer the missing block in two ways:

\[
p(x_1,x_2)
\;\longrightarrow\;
p(x_1\mid x_2)
\]

1. sample the conditional posterior with PyMC;
2. rewrite the same joint Gaussian as a linear Gaussian system and compute the posterior exactly.

Answers marked **[GIVEN]** are supplied material that should remain visible in the student self-work version.


## 0. Setup

**[GIVEN]**

The notebook targets the current course-facing PyMC/ArviZ stack used in this draft:

- PyMC 6.3.2
- arviz-base 1.3.0
- arviz-stats 1.3.2

The course repository should ultimately pin and validate these versions (or their replacements) before publication.


In [1]:
%pip install -q \
    "pymc==6.3.2" \
    "arviz-base==1.3.0" \
    "arviz-stats==1.3.2"


/home/opher/Repositories/ProbabilisticMachineLearning/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz_base as azb
import arviz_stats as azs

RANDOM_SEED = 2027
rng = np.random.default_rng(RANDOM_SEED)

print("PyMC:", pm.__version__)


ModuleNotFoundError: No module named 'pandas'

### The joint Gaussian

**[GIVEN]**

We use

\[
x=
\begin{pmatrix}
x_1\\
x_2
\end{pmatrix},
\]

with

\[
x_1=
\begin{pmatrix}
x_{11}\\x_{12}\\x_{13}
\end{pmatrix}
\in\mathbb R^3,
\qquad
x_2=
\begin{pmatrix}
x_{21}\\x_{22}\\x_{23}\\x_{24}
\end{pmatrix}
\in\mathbb R^4.
\]

The mean is

\[
\mu=
\begin{pmatrix}
\mu_1\\
\mu_2
\end{pmatrix},
\]

with

\[
\mu_1=(-0.5,0.5,1.0)^T,
\qquad
\mu_2=(0,0.5,-0.5,1.0)^T.
\]

The synthetic covariance construction is only a device for producing a controlled example. We use

\[
\sigma_1=1,\quad
\sigma_2=1.2,\quad
\rho_1=0.70,\quad
\rho_2=0.80,\quad
\kappa=0.90,
\]

with

\[
\Sigma_{11}
=
\sigma_1^2[(1-\rho_1)I_3+\rho_1J_3],
\]

\[
\Sigma_{22}
=
\sigma_2^2[(1-\rho_2)I_4+\rho_2J_4],
\]

and

\[
\Sigma_{12}
=
\kappa\sigma_1\sigma_2\sqrt{\rho_1\rho_2}\,
\mathbf 1_3\mathbf 1_4^T,
\qquad
\Sigma_{21}=\Sigma_{12}^T.
\]


In [ ]:
n1 = 3
n2 = 4

mu1 = np.array([-0.5, 0.5, 1.0])
mu2 = np.array([0.0, 0.5, -0.5, 1.0])
mu = np.concatenate([mu1, mu2])

sigma1 = 1.0
sigma2 = 1.2
rho1 = 0.70
rho2 = 0.80
kappa = 0.90

Sigma11 = sigma1**2 * (
    (1 - rho1) * np.eye(n1)
    + rho1 * np.ones((n1, n1))
)

Sigma22 = sigma2**2 * (
    (1 - rho2) * np.eye(n2)
    + rho2 * np.ones((n2, n2))
)

Sigma12 = (
    kappa
    * sigma1
    * sigma2
    * np.sqrt(rho1 * rho2)
    * np.ones((n1, n2))
)

Sigma21 = Sigma12.T


## 1. Examine the covariance structure


### 1.1 What are the dimensions of the four covariance blocks?

The setup defines \(x_1\in\mathbb R^3\) and \(x_2\in\mathbb R^4\).

What are the dimensions of

\[
\Sigma_{11},\quad
\Sigma_{12},\quad
\Sigma_{21},\quad
\Sigma_{22}?
\]


\[
\Sigma_{11}:3\times3,\qquad
\Sigma_{12}:3\times4,
\]

\[
\Sigma_{21}:4\times3,\qquad
\Sigma_{22}:4\times4.
\]


### 1.2 Calculate one cross-group covariance and correlation

Using

\[
\Sigma_{12}
=
\kappa\sigma_1\sigma_2
\sqrt{\rho_1\rho_2}\,
\mathbf 1_3\mathbf 1_4^T,
\]

calculate both

\[
\operatorname{Cov}(X_{11},X_{21})
\]

and

\[
\operatorname{Corr}(X_{11},X_{21}).
\]


Every entry of \(\Sigma_{12}\) is

\[
0.90(1)(1.2)\sqrt{0.70\cdot0.80}
\approx0.808.
\]

Therefore

\[
\operatorname{Cov}(X_{11},X_{21})\approx0.808.
\]

The corresponding correlation is

\[
\frac{0.808}{1(1.2)}
\approx0.673.
\]


### 1.3 Construct the full covariance matrix

Using `np.block`, construct

\[
\Sigma=
\begin{pmatrix}
\Sigma_{11}&\Sigma_{12}\\
\Sigma_{21}&\Sigma_{22}
\end{pmatrix}.
\]

Print its dimensions and check that they make sense given the dimensions of \(x_1\) and \(x_2\).


In [ ]:
Sigma = np.block([
    [Sigma11, Sigma12],
    [Sigma21, Sigma22],
])

print(Sigma.shape)


The result should have shape

\[
(7,7),
\]

because \(x\in\mathbb R^{3+4}=\mathbb R^7\).


### 1.4 Before plotting, what correlation structure should we see?

What should the off-diagonal correlation be for:

- two distinct components within \(x_1\);
- two distinct components within \(x_2\);
- one component from \(x_1\) and one from \(x_2\)?


Within \(x_1\),

\[
\rho_1=0.70.
\]

Within \(x_2\),

\[
\rho_2=0.80.
\]

Across groups,

\[
\kappa\sqrt{\rho_1\rho_2}
\approx0.673.
\]


**[GIVEN]** Run the following cell to inspect the correlation matrix.


In [ ]:
sd = np.sqrt(np.diag(Sigma))
Corr = Sigma / np.outer(sd, sd)

plt.figure(figsize=(6, 5))
plt.imshow(Corr, vmin=-1, vmax=1)
plt.colorbar(label="correlation")
plt.axvline(n1 - 0.5)
plt.axhline(n1 - 0.5)
plt.xticks(range(n1 + n2))
plt.yticks(range(n1 + n2))
plt.title("Correlation structure of the joint Gaussian")
plt.tight_layout()
plt.show()


## 2. Turn this into a learning problem


### 2.1 Generate one complete observation

**[GIVEN]**


In [ ]:
x_complete = rng.multivariate_normal(
    mean=mu,
    cov=Sigma,
)

x1_true = x_complete[:n1]
x2_obs = x_complete[n1:]

print("Hidden x1:", np.round(x1_true, 3))
print("Observed x2:", np.round(x2_obs, 3))


With the fixed random seed, the values should be approximately

\[
x_{1,\mathrm{true}}
=(-0.981,-0.275,1.892)
\]

and

\[
x_{2,\mathrm{obs}}
=(-0.184,-0.070,-0.078,0.919).
\]

We retain `x1_true` only so that we can evaluate the reconstruction later. It will not be used in inference.


### 2.2 What did the two slices do?

Explain the roles of

```python
x_complete[:n1]
```

and

```python
x_complete[n1:]
```

in the learning problem we are constructing.


`x_complete[:n1]` selects the first three components, the block \(x_1\). We save those values as `x1_true` and then treat them as hidden.

`x_complete[n1:]` selects the last four components, the block \(x_2\). Those values remain observed and will be used to infer \(x_1\).


### 2.3 Which machine-learning task is this?

Lecture 1 describes, among others,

\[
\text{prediction:}\qquad p_D(y_*\mid x_*),
\]

and

\[
\text{discovery:}\qquad p_D(z_*\mid x_*),
\]

where \(z\) can represent latent structure, including a missing value.

Which task class best describes reconstructing the hidden part of our vector from the observed part?


This is a **discovery** problem.

The quantity being inferred is latent rather than a separately supplied target label.


### 2.4 Apply the task schema to this example

Identify the generic \(z_*\) and \(x_*\) with variables in our example and write the conditional distribution that we want to infer.

The Gaussian parameters \(\mu\) and \(\Sigma\) are known in this controlled example.


Here,

\[
z_*=x_1,
\qquad
x_*=x_2.
\]

The concrete inference target is therefore

\[
\boxed{p(x_1\mid x_2,\mu,\Sigma)}.
\]


## 3. Infer the missing block with PyMC


### 3.1 Create the partially observed vector

**[GIVEN]**


In [ ]:
x_partial = x_complete.copy()
x_partial[:n1] = np.nan

x_partial


### 3.2 What does PyMC do with missing observed values?

Consult the current PyMC **Introductory Overview**, in its discussion of missing values in observed stochastic variables:

https://www.pymc.io/projects/docs/en/stable/pymc-examples/examples/getting_started.html

According to the documentation, what happens to missing entries supplied to an observed PyMC random variable?


PyMC treats the missing entries as unobserved random variables and samples them conditional on the observed entries.

For this model, the missing entries are the three components of \(x_1\), so the target is

\[
p(x_1\mid x_2,\mu,\Sigma).
\]


### 3.3 Extend an existing PyMC model

**[GIVEN]**

A PyMC `Model` is a container. We can create it first and later re-enter

```python
with model:
    ...
```

to add variables to the same model.


In [ ]:
model = pm.Model()


Use

```python
pm.MvNormal(
    name,
    mu=...,
    cov=...,
    observed=...,
)
```

to extend `model` with a multivariate-normal variable named `"x"` having mean `mu`, covariance `Sigma`, and observed value `x_partial`.


In [ ]:
with model:
    x = pm.MvNormal(
        "x",
        mu=mu,
        cov=Sigma,
        observed=x_partial,
    )


### 3.4 Sample and inspect the object PyMC now returns

Sample from `model` into a variable called `idata`.

Then:

1. use `type(idata)` to determine what kind of object the current PyMC stack returns;
2. inspect `idata["posterior"].data_vars` to see which variables are stored in the posterior.


In [ ]:
with model:
    idata = pm.sample(
        draws=2000,
        tune=1000,
        chains=4,
        random_seed=RANDOM_SEED,
        nuts={"target_accept": 0.9},
    )

print(type(idata))
print(idata["posterior"].data_vars)


For the target PyMC 6.3.2 / ArviZ 1.x stack, `pm.sample` should return an `xarray.DataTree`.

The posterior should contain an automatically created missing-value variable named `x_unobserved` (and may also contain a reconstructed representation of `x`, depending on the exact PyMC conversion behavior).

The notebook should be executed in the final course environment before publication so these expected names and types are validated rather than merely assumed.


### 3.5 Summarize the missing-value posterior with ArviZ

Use `azs.summary` to summarize `x_unobserved`.

Request:

- posterior mean;
- posterior SD;
- 90% HDI;
- `kind="stats"` because diagnostics are not the point of this exercise.


In [ ]:
pymc_summary = azs.summary(
    idata,
    var_names=["x_unobserved"],
    ci_prob=0.90,
    ci_kind="hdi",
    kind="stats",
)

pymc_summary


### 3.6 Display the PyMC posterior

**[GIVEN]**

`arviz_base.extract` combines the sampling dimensions and extracts the posterior variable. The remaining xarray manipulation is supplied plotting infrastructure, not student work.


In [ ]:
x1_posterior = azb.extract(
    idata,
    var_names="x_unobserved",
)

sample_dim = "sample"
component_dim = next(
    dim for dim in x1_posterior.dims
    if dim != sample_dim
)

posterior_samples = (
    x1_posterior
    .transpose(sample_dim, component_dim)
    .values
)

positions = np.arange(n1) + 1

def plot_component_posteriors(samples, truth, title):
    plt.figure(figsize=(8, 4.5))
    plt.violinplot(
        [samples[:, i] for i in range(samples.shape[1])],
        positions=positions,
        showmeans=True,
        showextrema=False,
    )
    plt.scatter(
        positions,
        truth,
        marker="D",
        s=45,
        label="Hidden true value",
    )
    plt.xticks(
        positions,
        [f"x1[{i}]" for i in range(n1)],
    )
    plt.ylabel("value")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_component_posteriors(
    posterior_samples,
    x1_true,
    "PyMC posterior",
)


## 4. Rewrite the same Gaussian as a linear Gaussian system

We are now changing notation.

The **partitioned-Gaussian notation** used so far is

\[
x_1,\ x_2,\ \mu_1,\ \mu_2,\ 
\Sigma_{11},\Sigma_{12},\Sigma_{21},\Sigma_{22}.
\]

The **linear Gaussian system (LGS)** notation from lecture is

\[
z\sim\mathcal N(\mu_z,\Sigma_z),
\]

\[
y\mid z\sim\mathcal N(Wz+b,\Omega).
\]

Once we establish the mapping, this section will stay in LGS notation.


### 4.1 Map the variables between the two notational systems

Which variables from the missing-data problem should correspond to \(z\) and \(y\)?

What are therefore \(\mu_z\) and \(\Sigma_z\)?


The hidden block is the latent state:

\[
z=x_1.
\]

The observed block is the observation:

\[
y=x_2.
\]

Therefore,

\[
\mu_z=\mu_1,
\qquad
\Sigma_z=\Sigma_{11}.
\]


In [ ]:
mu_z = mu1
Sigma_z = Sigma11
y = x2_obs


### 4.2 Find \(W\) and \(b\)

To construct the LGS observation model \(p(y\mid z)\), start from the partitioned-Gaussian conditional mean

\[
E[x_2\mid x_1]
=
\mu_2+
\Sigma_{21}\Sigma_{11}^{-1}(x_1-\mu_1).
\]

Substitute

\[
z=x_1,\qquad y=x_2,
\]

and rewrite the result in the LGS form

\[
E[y\mid z]=Wz+b.
\]

What are \(W\) and \(b\)?


\[
\boxed{
W=\Sigma_{21}\Sigma_{11}^{-1}
}
\]

and

\[
\boxed{
b=\mu_2-W\mu_1
}.
\]


### 4.3 Implement \(W\)

Use `np.linalg.solve(A, B)` to evaluate \(A^{-1}B\) without explicitly constructing the inverse.

Implement

\[
W=\Sigma_{21}\Sigma_{11}^{-1}.
\]


In [ ]:
W = np.linalg.solve(
    Sigma11,
    Sigma12,
).T

W


### 4.4 Implement \(b\)

Use the expression from Question 4.2.


In [ ]:
b = mu2 - W @ mu1
b


### 4.5 Write the equation for \(\Omega\)

The LGS covariance \(\Omega\) is

\[
\Omega=\operatorname{Cov}(y\mid z).
\]

Using the conditional-Gaussian covariance formula from lecture and the mapping

\[
y=x_2,\qquad z=x_1,
\]

write \(\Omega\) in terms of the original Gaussian covariance blocks.


\[
\boxed{
\Omega
=
\Sigma_{22}
-
\Sigma_{21}\Sigma_{11}^{-1}\Sigma_{12}
}.
\]


### 4.6 Implement \(\Omega\)

Use `np.linalg.solve` rather than explicitly constructing \(\Sigma_{11}^{-1}\).


In [ ]:
Omega = (
    Sigma22
    - Sigma21
    @ np.linalg.solve(
        Sigma11,
        Sigma12,
    )
)

Omega


### 4.7 Check that the LGS reproduces the original observed marginal

We now remain in **LGS notation**.

For a linear Gaussian system,

\[
m=W\mu_z+b
\]

and

\[
S=W\Sigma_zW^T+\Omega.
\]

Compute `m` and `S`.

Then use `np.allclose(a, b)` to check whether

\[
m=\mu_2
\]

and

\[
S=\Sigma_{22}.
\]

Why are those the equalities we expect?


In [ ]:
m = W @ mu_z + b

S = (
    W
    @ Sigma_z
    @ W.T
    + Omega
)

print("Mean check:", np.allclose(m, mu2))
print("Covariance check:", np.allclose(S, Sigma22))


The LGS was obtained by rewriting the original joint Gaussian. Its marginal distribution for

\[
y=x_2
\]

must therefore equal the original Gaussian marginal

\[
x_2\sim\mathcal N(\mu_2,\Sigma_{22}).
\]

That is why we expect

\[
m=\mu_2,
\qquad
S=\Sigma_{22}.
\]


**[GIVEN] Notation checkpoint**

The Lecture 1 conditioning section calls

\[
x_2-\mu_2
\]

the **innovation**.

After the LGS translation,

\[
y=x_2
\]

and

\[
m=\mu_2,
\]

so the same quantity is written

\[
y-m.
\]

No new mathematical object has been introduced; only the notation changed.


## 5. Compute the exact LGS posterior


### 5.1 Posterior equations

**[GIVEN]**

The lecture gives

\[
\mu_{z\mid y}
=
\mu_z+
\Sigma_zW^TS^{-1}(y-m),
\]

and

\[
\Sigma_{z\mid y}
=
\Sigma_z-
\Sigma_zW^TS^{-1}W\Sigma_z.
\]

We will apply these equations directly.


### 5.2 Implement the exact posterior mean

Use `np.linalg.solve` rather than explicitly constructing \(S^{-1}\).


In [ ]:
lgs_mean = (
    mu_z
    + Sigma_z
    @ W.T
    @ np.linalg.solve(
        S,
        y - m,
    )
)

lgs_mean


### 5.3 Implement the exact posterior covariance


In [ ]:
lgs_cov = (
    Sigma_z
    - Sigma_z
    @ W.T
    @ np.linalg.solve(
        S,
        W @ Sigma_z,
    )
)

lgs_cov


### 5.4 Look at the exact result

**[GIVEN]**


In [ ]:
lgs_sd = np.sqrt(np.diag(lgs_cov))

print("Posterior mean:")
print(np.round(lgs_mean, 3))

print("Posterior SD:")
print(np.round(lgs_sd, 3))

print("Posterior covariance:")
print(np.round(lgs_cov, 3))


The exact posterior mean should be approximately

\[
(-0.568,\ 0.432,\ 0.932),
\]

and each marginal posterior SD should be approximately

\[
0.683.
\]


### 5.5 Display the exact posterior

**[GIVEN]**


In [ ]:
lgs_plot_samples = rng.multivariate_normal(
    mean=lgs_mean,
    cov=lgs_cov,
    size=posterior_samples.shape[0],
)

plot_component_posteriors(
    lgs_plot_samples,
    x1_true,
    "Exact linear-Gaussian posterior",
)


## 6. Compare the two solutions


### 6.1 Make a numerical comparison

Use the PyMC summary from Section 3 and the exact LGS result to construct a table containing, for each hidden coordinate:

- PyMC posterior mean;
- exact LGS posterior mean;
- PyMC posterior SD;
- exact LGS posterior SD.


In [ ]:
comparison = pd.DataFrame({
    "PyMC mean": pymc_summary["mean"].to_numpy(),
    "LGS mean": lgs_mean,
    "PyMC SD": pymc_summary["sd"].to_numpy(),
    "LGS SD": lgs_sd,
})

comparison.index = [
    f"x1[{i}]"
    for i in range(n1)
]

comparison


The PyMC values should be close to the exact LGS values. The remaining differences are Monte Carlo error in the sampled PyMC summaries.


### 6.2 Display the two posterior means together

**[GIVEN]**


In [ ]:
plot_positions = np.arange(n1)
offset = 0.07

plt.figure(figsize=(8, 4.5))

plt.errorbar(
    plot_positions - offset,
    pymc_summary["mean"].to_numpy(),
    yerr=pymc_summary["sd"].to_numpy(),
    fmt="o",
    capsize=4,
    label="PyMC mean ± SD",
)

plt.errorbar(
    plot_positions + offset,
    lgs_mean,
    yerr=lgs_sd,
    fmt="o",
    capsize=4,
    label="Exact LGS mean ± SD",
)

plt.scatter(
    plot_positions,
    x1_true,
    marker="D",
    s=45,
    label="Hidden true value",
)

plt.xticks(
    plot_positions,
    [f"x1[{i}]" for i in range(n1)],
)

plt.ylabel("value")
plt.title("Posterior comparison")
plt.legend()
plt.tight_layout()
plt.show()


## What this notebook established

**[GIVEN]**

We began with one joint Gaussian,

\[
p(x_1,x_2),
\]

and hid \(x_1\).

PyMC then sampled the conditional posterior

\[
p(x_1\mid x_2,\mu,\Sigma).
\]

We next rewrote the same joint Gaussian as the linear Gaussian system

\[
z\sim\mathcal N(\mu_z,\Sigma_z),
\qquad
y\mid z\sim\mathcal N(Wz+b,\Omega),
\]

with

\[
z=x_1,\qquad y=x_2.
\]

Applying the exact LGS posterior equations produced the same conditional distribution, up to Monte Carlo error in the PyMC approximation.
